# Chapter 8 — Linking Ontologies to Data
### Notebook 0 · Overview and setup

*Book reference: Keet, *Ontology Engineering* (2nd ed.), Ch. 8*

Every chapter so far has assumed the data was already in the ontology's vocabulary. It never is. Chapter 8 is about the gap: the data sits in a relational database that has never heard of your classes, and the questions are asked in terms that exist only in your ontology.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch08_toolkit as ch8
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
import oe_course; print(json.dumps(oe_course.describe_environment(), indent=1))

## Notebooks in this chapter

| # | Notebook | Book section | What you build |
|---|---|---|---|
| 0 | `00_overview_and_setup` | — | environment check |
| 1 | `01_mappings_and_materialisation` | 8.1–8.2 | R2RML-style mappings + an ETL pipeline |
| 2 | `02_query_rewriting` | 8.3 | **a query rewriter** — SPARQL to SQL |
| 3 | `03_exercises` | 8.4 | autograded answers |
| 4 | `04_agentic_lab` | — | a mapping agent + the **materialise-or-rewrite MDP** |


**By the end of this notebook you can:**

1. Write mappings from a relational schema to an ontology, and say why a foreign key must become an IRI.
2. Implement **both** OBDA strategies and check they agree — the property that makes the approach trustworthy.
3. Read the SQL a conjunctive query rewrites to, and explain where the ontology went.
4. Price the materialise-vs-rewrite decision, including the cost of **staleness**.

## The two strategies

| | materialisation | query rewriting |
|---|---|---|
| when | run the mapping once, up front | translate each query at run time |
| stores | a copy of everything, as triples | nothing |
| query cost | low | a join per atom |
| freshness | **stale between refreshes** | always current |
| §8.2 calls it | ETL / warehousing | virtual / on-the-fly |

Both are implemented in `ch08_toolkit` over the *same* mappings, and Notebook 2 checks that they return identical answers. Once that holds, the choice is an engineering trade rather than a matter of taste — and Notebook 4 optimises it.

### Sanity check: a real database, a real graph

In [ ]:
conn = ch8.build_database()
counts = {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
          for t in ['ward', 'patient', 'diagnosis', 'code_lookup']}
print('source rows:', counts)
graph = ch8.materialise(conn)
print('materialised triples:', len(graph))
assert len(graph) > 30